# A Triton matmul: tl.dot, autotuning, grouped ordering and fused epilogues

Write a tensor-core matmul in about 40 lines of Triton, see how block shapes map to the tiling you did in CUDA, use the autotuner to pick block sizes, warps and pipeline depth per shape, reorder programs so tiles stay in L2, fuse an activation into the epilogue, and see why decode-sized matmuls need split-K or GEMV-style kernels.

This notebook holds the code from the lesson **[A Triton matmul: tl.dot, autotuning, grouped ordering and fused epilogues](https://inferenceacademy.com/learn/kern-triton/triton-matmul-autotuning-and-grouped-ordering)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## The kernel

In [ ]:
import torch, triton, triton.language as tl

@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64,  "BLOCK_K": 32, "GROUP_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 64,  "BLOCK_K": 32, "GROUP_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_M": 32,  "BLOCK_N": 64,  "BLOCK_K": 32, "GROUP_M": 8}, num_stages=5, num_warps=2),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                  stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                  BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
                  GROUP_M: tl.constexpr, ACTIVATION: tl.constexpr):
    # 1. Which output tile is mine? (grouped ordering, explained below)
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # 2. Pointers to the first BLOCK_K slices of A and B for this tile.
    offs_m = (pid_m * BLOCK_M + tl.arange(0, BLOCK_M)) % M
    offs_n = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    # 3. Walk along K, accumulating in FP32 on tensor cores.
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_K, other=0.0)
        acc = tl.dot(a, b, acc)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # 4. Epilogue: optional fused activation, then store.
    if ACTIVATION == "silu":
        acc = acc * tl.sigmoid(acc)
    c = acc.to(tl.float16)
    offs_cm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + offs_cm[:, None] * stride_cm + offs_cn[None, :] * stride_cn
    tl.store(c_ptrs, c, mask=(offs_cm[:, None] < M) & (offs_cn[None, :] < N))

def matmul(a, b, activation=""):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]) * triton.cdiv(N, meta["BLOCK_N"]),)
    matmul_kernel[grid](a, b, c, M, N, K, a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                        c.stride(0), c.stride(1), ACTIVATION=activation)
    return c

### Test and time it

In [ ]:
a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
b = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
torch.testing.assert_close(matmul(a, b), a @ b, atol=1e-2, rtol=1e-2)
for name, f in [("triton", lambda: matmul(a, b)), ("cuBLAS", lambda: a @ b)]:
    ms = triton.testing.do_bench(f)
    print(f"{name}: {2 * 4096**3 / (ms / 1e3) / 1e12:.1f} TFLOP/s")

## Autotuning: what the knobs mean

## Grouped ordering: making L2 do the work

## Fused epilogues

## Decode shapes: when tiles don't fit

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-triton/triton-matmul-autotuning-and-grouped-ordering